# T10 — ARIMA Model (AutoRegressive Integrated Moving Average) — v3

**Input:** `data/processed/{train,test}_features.csv`  
**Goal:** predict RUL by forecasting the health-index with ARIMA(p,d,q) until it crosses the failure threshold.

### Why ARIMA is the natural choice for RUL
The health-index is **non-stationary** — it has a persistent upward trend as the engine degrades. AR/ARMA assume stationarity, so a plain forecast will eventually revert to a stationary mean. ARIMA's differencing (`d ≥ 1`) removes the unit root; with `trend='c'` on the differenced series we retain a linear drift when integrating back, which is exactly the behaviour we want in a degradation forecast.

This notebook also aggregates the T08, T09, T10 predictions into a final classical-model comparison table.

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from itertools import product
from statsmodels.tsa.arima.model import ARIMA as StatsARIMA
from statsmodels.tsa.stattools import adfuller
warnings.filterwarnings("ignore")

from src.models.classical import (
    build_pca_health_index,
    compute_failure_threshold,
    predict_rul_arima,
    predict_dataset,
    simulate_test_from_train,
    smooth_series,
    RUL_CAP,
)
from src.evaluation.metrics import evaluate, evaluate_per_subset, summarise_all_models

PROC_DIR    = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_COLS = [f"s{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

## 1. Load & prepare

In [ ]:
train = pd.read_csv(PROC_DIR / "train_features.csv")
test  = pd.read_csv(PROC_DIR / "test_features.csv")
print(f"train: {train.shape}  |  test: {test.shape}")

train, test = build_pca_health_index(train, test, SENSOR_COLS)
THRESHOLD = compute_failure_threshold(train, end_of_life_rul=5, quantile=0.5)
print(f"failure threshold: {THRESHOLD:.3f}")

## 2. Stationarity check — Augmented Dickey-Fuller
If the level series is non-stationary (p > 0.05) and its first difference is stationary, then `d = 1` is the right differencing order.

In [ ]:
sample = train[train.dataset_id == 1].engine_id.unique()[:6]
print(f"{'engine':<8}{'level p':<12}{'diff-1 p':<12}{'recommended d'}")
print("-" * 48)
for eid in sample:
    s = train[train.engine_id == eid].sort_values("cycle").health_index.values
    p0 = adfuller(s)[1]
    p1 = adfuller(np.diff(s))[1]
    d  = 0 if p0 < 0.05 else (1 if p1 < 0.05 else 2)
    print(f"{eid:<8}{p0:<12.4f}{p1:<12.4f}d={d}")
print("\nlevel p > 0.05 AND diff-1 p < 0.05  ⇒  d = 1 is correct")

## 3. Demo: ARIMA forecast on one test engine

In [ ]:
eid = test[test.dataset_id == 1].engine_id.iloc[0]
g   = test[test.engine_id == eid].sort_values("cycle")
raw    = g.health_index.values
smooth = smooth_series(raw, window=5)

model = StatsARIMA(smooth, order=(2, 1, 2), trend="t").fit()
fcst  = model.forecast(steps=100)

plt.figure(figsize=(10, 4))
plt.plot(range(len(raw)),    raw,    color="lightgray", label="raw")
plt.plot(range(len(smooth)), smooth, color="steelblue", label="smoothed (history)")
plt.plot(range(len(smooth), len(smooth) + len(fcst)), fcst, color="darkorange", label="ARIMA(2,1,2) forecast")
plt.axhline(THRESHOLD, color="red", ls="--", label=f"threshold={THRESHOLD:.2f}")
plt.xlabel("cycle (from start of test)"); plt.ylabel("health_index")
plt.title(f"ARIMA forecast — engine {eid}  (true RUL = {int(g.RUL.iloc[-1])})")
plt.legend(); plt.tight_layout(); plt.show()

## 4. (p, d, q) grid search on a simulated validation set
We fix `d=1` from the ADF analysis and vary `p` and `q`.

In [ ]:
val_df = simulate_test_from_train(train, cutoff_fraction=0.6, max_engines=100, random_seed=42)
print(f"simulated-val engines: {val_df.engine_id.nunique()}")

rows = []
for p, d, q in product([1, 2, 3], [1], [1, 2, 3]):
    fn = partial(predict_rul_arima, threshold=THRESHOLD, p=p, d=d, q=q)
    yt, yp, _ = predict_dataset(val_df, fn)
    m = evaluate(yt, yp, model_name=f"ARIMA({p},{d},{q})", verbose=False)
    rows.append({"p": p, "d": d, "q": q, **m})

tune_df = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
print(tune_df.to_string(index=False))

BEST_P = int(tune_df.iloc[0].p)
BEST_D = int(tune_df.iloc[0].d)
BEST_Q = int(tune_df.iloc[0].q)
print(f"\nBest order: ARIMA({BEST_P},{BEST_D},{BEST_Q})")

## 5. Final evaluation on official test set

In [ ]:
predict_fn = partial(predict_rul_arima, threshold=THRESHOLD, p=BEST_P, d=BEST_D, q=BEST_Q)
y_true, y_pred, dids = predict_dataset(test, predict_fn)

print(f"=== ARIMA({BEST_P},{BEST_D},{BEST_Q}) — Test Results ===")
test_metrics = evaluate_per_subset(y_true, y_pred, dids, model_name="ARIMA")
display(test_metrics)

## 6. Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(y_true, y_pred, alpha=0.4, s=15, color="steelblue")
axes[0].plot([0, RUL_CAP], [0, RUL_CAP], "r--")
axes[0].set_xlabel("true RUL"); axes[0].set_ylabel("predicted RUL")
axes[0].set_title(f"ARIMA({BEST_P},{BEST_D},{BEST_Q}) — Pred vs True")

err = y_pred - y_true
axes[1].hist(err, bins=40, color="coral", edgecolor="white")
axes[1].axvline(0, color="black", ls="--")
axes[1].axvline(err.mean(), color="red", label=f"mean={err.mean():+.1f}")
axes[1].set_xlabel("error (pred-true)"); axes[1].set_title("error distribution"); axes[1].legend()

axes[2].hist(y_pred, bins=40, color="mediumpurple", edgecolor="white")
axes[2].set_xlabel("predicted RUL"); axes[2].set_title("prediction distribution")
n_cap = int((y_pred >= RUL_CAP).sum())
print(f"engines predicted at cap: {n_cap}/{len(y_pred)}")
plt.tight_layout(); plt.show()

## 7. Save predictions

In [ ]:
out = pd.DataFrame({
    "model":      "ARIMA",
    "p":          BEST_P,
    "d":          BEST_D,
    "q":          BEST_Q,
    "y_true":     y_true,
    "y_pred":     y_pred,
    "dataset_id": dids,
})
out.to_csv(RESULTS_DIR / "T10_ARIMA_predictions.csv", index=False)

overall = evaluate(y_true, y_pred, model_name="ARIMA_test", verbose=False)
print(f"saved | RMSE={overall['rmse']:.4f}  NASA={overall['nasa_score']:.2f}")

## 8. Classical-model comparison (T08 + T09 + T10)
Aggregates the saved predictions into a single ranked table. **Run T08 and T09 first** so their CSVs exist.

In [ ]:
all_results = {}
for tag, fname in [
    ("AR",    "T08_AR_predictions.csv"),
    ("ARMA",  "T09_ARMA_predictions.csv"),
    ("ARIMA", "T10_ARIMA_predictions.csv"),
]:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        m = evaluate(df.y_true.values, df.y_pred.values, model_name=tag, verbose=False)
        all_results[tag] = m
    else:
        print(f"  [WARN] {fname} not found — run T08/T09 first")

if all_results:
    comparison = summarise_all_models(all_results)
    print("\n=== Classical Model Comparison (Test Set) ===")
    display(comparison)